# QLoRA Fine-Tuning — Suporte Técnico (Google Colab)

Notebook para treinar o modelo OPT-125M com QLoRA em GPU e baixar os adaptadores treinados.

In [ ]:
!pip install -q torch transformers peft bitsandbytes trl datasets accelerate

In [ ]:
import json
import torch
from pathlib import Path
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
TRAIN_DATA_JSONL = """{"prompt": "Meu computador está muito lento, como posso melhorar seu desempenho?", "response": "Você pode começar desinstalando programas que não usa mais. Além disso, verifique se há atualizações do sistema operacional e do software. Execute uma verificação completa de vírus e limpe os arquivos temporários usando a ferramenta de limpeza de disco. Também considere adicionar mais memória RAM se o problema persistir."}
{"prompt": "Preciso de ajuda para configurar minha VPN.", "response": "Verifique se você tem as credenciais corretas, como nome de usuário, senha e endereço do servidor da VPN. Abra o aplicativo cliente da VPN em seu dispositivo e insira essas informações. Depois, conecte-se e, se encontrar algum erro, consulte a documentação do serviço de VPN ou entre em contato com o suporte técnico."}
{"prompt": "Minha impressora não está respondendo, o que posso fazer?", "response": "Verifique se a impressora está ligada e conectada à mesma rede que seu computador. Tente reiniciar a impressora e o computador. Verifique se há atolamentos de papel ou se os níveis de tinta estão baixos. Se os problemas persistirem, remova a impressora da lista de dispositivos e adicione-a novamente."}
{"prompt": "Estou recebendo um erro ao tentar abrir um programa, o que pode ser?", "response": "Isso pode ser causado por uma instalação corrompida ou um conflito com outro software. Tente reinstalar o programa. Se o erro persistir, faça uma verificação completa do sistema em busca de vírus e atualize os drivers de hardware, se necessário."}
{"prompt": "Esqueci minha senha de e-mail, como posso recuperá-la?", "response": "Acesse a página de login do seu provedor de e-mail e clique em 'Esqueci minha senha'. Siga as instruções para receber um link de redefinição de senha no seu número de telefone ou e-mail alternativo cadastrado. Se não conseguir, entre em contato com o suporte do provedor de e-mail para assistência adicional."}
{"prompt": "Minha impressora não está imprimindo, o que devo fazer?", "response": "Primeiro, verifique se a impressora está ligada e conectada ao computador. Veja se há papel e tinta. Em seguida, tente reiniciar a impressora e o computador. Se o problema persistir, verifique se a impressora está definida como o dispositivo padrão."}
{"prompt": "Minha impressora não está imprimindo, o que fazer?", "response": "Verifique se a impressora está conectada corretamente e se há papel e tinta. Abra o painel de controle da impressora no seu computador para verificar se ela está definida como a impressora padrão e se há algum trabalho de impressão enfileirado."}
{"prompt": "Esqueci minha senha do e-mail, como posso recuperá-la?", "response": "Acesse a página de login do seu provedor de e-mail e clique em 'Esqueci minha senha'. Siga as instruções enviadas para o seu número de telefone ou outro e-mail, se você tiver configurado isso previamente."}
{"prompt": "Meu sistema operacional não está atualizando, o que fazer?", "response": "Verifique se você tem espaço suficiente em disco para a atualização. Tente reiniciar o computador e atualize manualmente através das configurações. Se o problema continuar, consulte o suporte técnico para possíveis soluções mais avançadas."}
{"prompt": "Estou tendo problemas com a conexão VPN, o que posso fazer?", "response": "Verifique se você está utilizando as credenciais corretas e se a conexão com a internet está estável. Tente desconectar e reconectar à VPN. Se o problema persistir, reinicie o computador e verifique se há atualizações disponíveis para o software da VPN."}
{"prompt": "Não consigo fazer backup dos meus dados, o que está acontecendo?", "response": "Verifique se o dispositivo de backup está corretamente conectado e reconhecido pelo seu computador. Certifique-se de ter espaço suficiente no dispositivo. Caso esteja usando um software específico, verifique se está atualizado e configurado corretamente."}
{"prompt": "Esqueci minha senha de e-mail, como posso recuperá-la?", "response": "Acesse a página de login do seu provedor de e-mail e clique em 'Esqueci minha senha'. Siga as instruções enviadas para o seu número de telefone ou e-mail alternativo para redefinir sua senha."}
{"prompt": "Como faço backup dos meus dados?", "response": "Você pode fazer backup dos seus dados usando um disco rígido externo ou serviços de armazenamento em nuvem, como Google Drive ou Dropbox. Selecione as pastas e arquivos que deseja salvar e copie-os para o local escolhido. Para backups automáticos, configure seu software de backup para realizar cópias regulares."}
{"prompt": "Como posso fazer backup dos meus dados importantes?", "response": "Você pode usar um disco rígido externo ou um serviço de armazenamento em nuvem, como Google Drive ou Dropbox. Para fazer o backup, conecte o disco rígido ao computador e copie e cole os arquivos importantes. Para serviços de nuvem, faça login na sua conta e carregue seus arquivos. Considere também programar backups automáticos se a ferramenta permitir."}
{"prompt": "Meu computador está com vírus, como posso resolver isso?", "response": "Instale um software antivírus confiável e execute uma verificação completa do sistema. Siga as instruções para remover qualquer ameaça detectada. Depois, atualize seu antivírus e faça verificações regulares para evitar infecções futuras. Além disso, evite clicar em links suspeitos e faça downloads apenas de fontes confiáveis."}
{"prompt": "Como configuro meu e-mail no aplicativo de celular?", "response": "Abra o aplicativo de e-mail e escolha a opção para adicionar uma nova conta. Insira seu endereço de e-mail e senha. Em muitos casos, o aplicativo irá detectar automaticamente as configurações. Se não, você precisará inserir manualmente as configurações de servidor IMAP ou POP3, que podem ser obtidas no site do seu provedor de e-mail."}
{"prompt": "Meu computador está mostrando um erro ao tentar abrir um programa, o que devo fazer?", "response": "Primeiro, tente reiniciar o computador e abrir o programa novamente. Se o erro persistir, verifique se há atualizações disponíveis para o software e instale-as. Caso o problema continue, você pode tentar desinstalar e reinstalar o programa. Se precisar de mais assistência, forneça o código de erro que aparece na tela."}
{"prompt": "Meu computador está infectado com um vírus, como posso removê-lo?", "response": "Instale um software antivírus de confiança e faça uma varredura completa no sistema. Após a varredura, siga as recomendações do antivírus para remover ameaças. Certifique-se também de manter o antivírus atualizado e evite baixar arquivos de fontes não confiáveis."}
{"prompt": "Meu mouse não está funcionando, o que pode ser?", "response": "Primeiro, verifique se ele está conectado corretamente. Tente conectar em outra porta USB. Se for um mouse sem fio, verifique a bateria. Você também pode testar em outro computador para ver se o problema é com o mouse ou com a porta."}
{"prompt": "Preciso fazer backup dos meus dados, como faço isso?", "response": "Você pode usar um serviço de armazenamento em nuvem, como Google Drive ou OneDrive, para fazer backup dos seus arquivos. Alternativamente, conecte um disco rígido externo ao seu computador e copie os arquivos importantes manualmente. Certifique-se de incluir documentos, fotos e outros dados essenciais."}
{"prompt": "Esqueci minha senha de e-mail e não consigo acessar minha conta.", "response": "Visite a página de login do seu provedor de e-mail e clique em 'Esqueci minha senha'. Siga as instruções, que geralmente incluem a verificação de identidade através de um código enviado para seu telefone ou outro e-mail. Se você não conseguir redefinir sua senha, entre em contato com o suporte do provedor de e-mail."}
{"prompt": "Meu Wi-Fi está muito instável, o que posso fazer?", "response": "Primeiro, reinicie seu roteador desconectando e ligando novamente. Verifique se outros dispositivos estão funcionando bem na rede. Se o problema persistir, tente mudar a posição do roteador ou diminuir a interferência de outros aparelhos eletrônicos."}
{"prompt": "Não consigo me conectar à minha VPN, o que pode estar errado?", "response": "Verifique sua conexão com a internet e se as credenciais da VPN estão corretas. Tente desconectar e reconectar ou reiniciar seu computador. Se ainda assim não funcionar, consulte o suporte de TI da sua empresa."}
{"prompt": "Meu computador está com vírus, como posso resolver isso?", "response": "Instale ou atualize seu software antivírus e execute uma verificação completa do sistema. Siga as instruções fornecidas pelo software para remover quaisquer ameaças detectadas. Além disso, considere usar um software anti-malware adicional para uma verificação mais profunda. Para garantir sua segurança, evite clicar em links ou baixar arquivos de fontes não confiáveis."}
{"prompt": "Meu computador está muito lento, como posso melhorar seu desempenho?", "response": "Você pode tentar as seguintes etapas: 1) Feche programas que não está utilizando. 2) Verifique se há atualizações disponíveis para o sistema operacional e programas. 3) Execute uma verificação de vírus. 4) Limpe arquivos temporários usando uma ferramenta como o 'Limpeza de Disco'. 5) Considere aumentar a memória RAM se o problema continuar."}
{"prompt": "Estou com problemas para conectar ao Wi-Fi, o que fazer?", "response": "Primeiro, verifique se o Wi-Fi está ativado no seu dispositivo. Em seguida, reinicie o roteador, desconectando-o da energia por 10 segundos. Depois disso, tente se conectar novamente. Se o problema persistir, verifique se há atualizações de driver de rede disponíveis."}
{"prompt": "Estou tendo dificuldades para conectar à minha VPN, o que pode ser o problema?", "response": "Verifique se suas credenciais de login estão corretas. Certifique-se de que a conexão de internet está funcionando. Tente desconectar e reconectar à VPN. Se o problema continuar, verifique se o software da VPN está atualizado e consulte o suporte do provedor da VPN para obter assistência específica."}
{"prompt": "Preciso fazer backup dos meus dados. Quais opções eu tenho?", "response": "Você pode usar um disco rígido externo ou serviços de armazenamento em nuvem, como Google Drive ou Dropbox. Certifique-se de fazer backups regulares, mantendo cópias de segurança dos arquivos mais importantes."}
{"prompt": "Meu mouse está travando e não responde corretamente.", "response": "Primeiro, verifique se as pilhas do mouse sem fio estão boas ou se o cabo do mouse com fio está bem conectado. Tente limpar o sensor na parte inferior do mouse, pois sujeira pode afetar o desempenho. Se ainda tiver problemas, experimente usar o mouse em outra superfície ou teste com outro computador para ver se o problema persiste."}
{"prompt": "Estou com problemas para conectar ao Wi-Fi, o que devo fazer?", "response": "Primeiro, verifique se o Wi-Fi está ativado no seu dispositivo. Em seguida, reinicie seu roteador desconectando da energia por 10 segundos. Depois, tente conectar novamente à rede. Se o problema persistir, acesse as configurações de rede e esqueça a rede atual, em seguida, reconecte-se digitando a senha novamente."}
{"prompt": "Meu computador está muito lento, como posso resolver isso?", "response": "Você pode começar fechando programas desnecessários que estão abertos. Além disso, faça uma verificação de vírus e malware. Por fim, considere desfragmentar o disco rígido ou adicionar mais memória RAM, se possível."}
{"prompt": "Meu computador está muito lento, o que posso fazer?", "response": "Tente fechar programas que você não está usando. Você também pode verificar se há atualizações do sistema operacional disponíveis. Além disso, execute uma verificação de vírus com seu software antivírus. Se o problema persistir, considere aumentar a memória RAM ou fazer uma limpeza de disco para remover arquivos desnecessários."}
{"prompt": "Estou recebendo um erro ao tentar abrir um programa, o que pode ser?", "response": "Isso pode acontecer devido a arquivos corrompidos ou falta de atualizações. Tente desinstalar e reinstalar o programa. Também verifique se existem atualizações disponíveis para ele e para o sistema operacional."}
{"prompt": "Minha conexão Wi-Fi está caindo constantemente, o que eu posso fazer?", "response": "Primeiro, verifique se outros dispositivos estão enfrentando o mesmo problema. Se sim, reinicie o roteador desconectando-o da energia por 10 segundos. Se o problema persistir, atualize o firmware do roteador e verifique se há interferências de outros dispositivos eletrônicos. Também é útil mudar o canal de transmissão nas configurações do roteador."}
{"prompt": "Meu mouse não está funcionando, o que fazer?", "response": "Verifique se o mouse está corretamente conectado ao computador. Tente desconectar e reconectar. Se for sem fio, verifique as pilhas. Teste o mouse em outro computador para determinar se o problema é com o mouse ou a porta do computador."}
{"prompt": "Estou preocupado com vírus no meu computador. O que devo fazer?", "response": "Instale um software antivírus se ainda não tiver um. Execute uma varredura completa do sistema. Mantenha o antivírus atualizado e evite acessar sites desconhecidos ou baixar arquivos de fontes não confiáveis."}
{"prompt": "Minha impressora não está respondendo.", "response": "Primeiro, verifique se a impressora está ligada e conectada ao computador ou à rede Wi-Fi. Tente reiniciar a impressora. Se estiver conectada via USB, experimente trocar a porta USB. Se a impressora for sem fio, verifique se está conectada à mesma rede Wi-Fi que o seu computador. Além disso, confira se há papel ou tinta suficientes."}
{"prompt": "Preciso recuperar minha senha, como faço isso?", "response": "Acesse a página de login e clique em 'Esqueci minha senha'. Siga as instruções, que geralmente incluem inserir seu e-mail para receber um link de redefinição de senha. Se não receber o e-mail, verifique sua pasta de spam."}
{"prompt": "Estou tendo problemas para conectar ao Wi-Fi, o que eu faço?", "response": "Primeiro, verifique se o Wi-Fi está ativado no seu dispositivo. Em seguida, tente reiniciar o roteador. Se o problema persistir, desconecte-se da rede Wi-Fi e reconecte-se, certificando-se de que a senha está correta. Se ainda não funcionar, verifique se outros dispositivos conseguem se conectar à mesma rede."}
{"prompt": "Estou recebendo um erro ao tentar abrir meu software de edição de imagens.", "response": "Verifique se o software está atualizado. Às vezes, uma simples atualização pode resolver problemas de compatibilidade. Se o erro persistir, tente desinstalar e reinstalar o software. Se você ainda encontrar o mesmo erro, consulte o site do fabricante para obter orientações específicas ou suporte técnico."}
{"prompt": "Meu sistema operacional precisa de uma atualização, como posso fazê-la?", "response": "Acesse as configurações do sistema e procure a opção de 'Atualização e Segurança'. Clique em 'Verificar atualizações' e siga as instruções para instalar as atualizações disponíveis. Reinicie o computador se solicitado."}
{"prompt": "Preciso fazer backup dos meus dados, qual é a melhor maneira de proceder?", "response": "Você pode usar um disco rígido externo ou um serviço de armazenamento em nuvem como Google Drive, Dropbox ou OneDrive. Para um backup rápido, copie os arquivos importantes manualmente para o dispositivo de armazenamento externo. Para um backup completo, utilize a ferramenta de backup do sistema operacional, que geralmente oferece opções para agendar backups automáticos."}
{"prompt": "Estou tendo problemas para conectar no Wi-Fi da minha casa.", "response": "Primeiro, verifique se o roteador está ligado e funcionando. Tente reiniciar o roteador desconectando-o da tomada por 10 segundos. Em seguida, certifique-se de que o Wi-Fi está ativado no seu dispositivo. Se ainda não conseguir conectar, tente esquecer a rede nas configurações de Wi-Fi e conecte novamente, inserindo a senha correta."}
{"prompt": "Estou tendo problemas para me conectar à VPN, o que fazer?", "response": "Verifique sua conexão à Internet e tente reconectar. Certifique-se de que as credenciais de login estão corretas. Se o problema persistir, entre em contato com o suporte de TI para verificar se há manutenção na VPN."}
{"prompt": "Recebi uma mensagem de erro ao tentar abrir um programa, o que isso significa?", "response": "Erros ao abrir programas podem ser causados por arquivos corrompidos ou falta de atualizações. Tente reinstalar o programa afetado. Verifique se há atualizações de software disponíveis e instale-as. Se o erro continuar, verifique se há dependências necessárias que estão faltando."}
{"prompt": "Esqueci minha senha de e-mail, como posso recuperá-la?", "response": "Acesse a página de login do seu provedor de e-mail e clique em 'Esqueci minha senha'. Siga as instruções fornecidas, que geralmente envolvem o envio de um código para o seu telefone ou outro e-mail associado, ou responder a perguntas de segurança."}
{"prompt": "Meu sistema operacional não está atualizando.", "response": "Verifique se você tem uma conexão de internet estável. Acesse as configurações de atualização do sistema e procure por atualizações manualmente. Caso não consiga atualizar, reinicie o computador e tente novamente. Se o problema persistir, pode ser necessário verificar se há espaço suficiente em disco ou reparar a instalação do sistema operacional."}
{"prompt": "Estou tendo dificuldades para imprimir, a impressora não responde.", "response": "Verifique se a impressora está ligada e conectada ao computador ou à rede. Tente reiniciar a impressora e o computador. Certifique-se de que não há filas de impressão bloqueadas no seu computador. Vá em 'Dispositivos e Impressoras', clique com o botão direito na impressora e selecione 'Ver o que está sendo impresso'. Se houver tarefas travadas, cancele-as."}
{"prompt": "Meu sistema operacional não está atualizando, como resolver?", "response": "Verifique se há espaço suficiente no disco. Tente reiniciar seu computador e, em seguida, forçar a atualização pela seção de configurações do sistema. Caso persista, consulte os logs de erro ou o suporte do fabricante."}"""

train_raw = [json.loads(line) for line in TRAIN_DATA_JSONL.strip().split("\n")]
print(f"Amostras de treino: {len(train_raw)}")

In [ ]:
MODEL_NAME = "facebook/opt-125m"
OUTPUT_DIR = "outputs/qlora-suporte-tecnico"

def format_chat(sample: dict) -> str:
    return (
        f"Sistema: Você é um assistente de suporte técnico de TI. "
        f"Responda de forma clara e profissional.\n"
        f"Usuário: {sample['prompt']}\n"
        f"Assistente: {sample['response']}"
    )

train_data = [{"text": format_chat(s)} for s in train_raw]
train_dataset = Dataset.from_list(train_data)
print(train_dataset)

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model)
print("Modelo carregado com quantização 4-bit")

In [ ]:
lora_config = LoraConfig(
    r=64,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    max_steps=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    max_grad_norm=0.3,
    max_length=128,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args,
)

trainer.train()

In [ ]:
adapter_path = f"{OUTPUT_DIR}/final_adapter"
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adaptadores salvos em: {adapter_path}")

import os
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f))
    print(f"  {f} ({size / 1024:.1f} KB)")

In [ ]:
import shutil
shutil.make_archive("final_adapter", "zip", adapter_path)

from google.colab import files
files.download("final_adapter.zip")
print("Download do adapter iniciado!")